# Model evaluation and comparison

**Data sources used in this notebook:**
- `sudan_results.csv` - current set of runs (k=0.5/1.0 only, post threshold-fix, active window extended to include 2025)
- `sudan_results_historical.csv` - earlier runs including k=0.25, used only for the k-selection comparison in Section 1
- `ethiopia_results.csv` — Ethiopia/Tigray replication runs

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.constants import ONSET_END_DATE, ONSET_START_DATE
from utils.data_prep import get_clean_combined_data

sudan = pd.read_csv("evaluation/sudan_results_combined.csv")
# sudan = pd.read_csv("evaluation/sudan_results.csv")
# sudan = pd.read_csv("evaluation/sudan_results_historical.csv") # Before the collapse
ethiopia = pd.read_csv("evaluation/ethiopia_results.csv")

print(f"Sudan: {len(sudan)} runs")
print(f"Ethiopia: {len(ethiopia)} runs")

Sudan: 1013 runs
Ethiopia: 32 runs


## 1. Choosing the k escalation threshold
`k` controls the escalation threshold — a lower k means a looser threshold (escalation is easier to trigger).

**Issue**

Raw AUPR favours the lowest `k`, but that is misleading for conflict prediction. The model was catching 100% of true positives by default rather than through genuine skill. Domain knowledge also matters here, in conflict forecasting, you want a model that's sensitive to real escalations without flagging every small, ordinary shift in conflict as significant.

**Fix**

Ten values of `k` (0.25 to 2.5) were tested directly, measuring both the true onset-window prevalence of escalation and the rate at which the F1-optimal threshold collapsed to predicting positive for nearly every region-month. Prevalence declines smoothly from `k`=0.25 to 14.4% at `k`=2.5, while collapse rate falls sharply and non-linearly — from 75.0% at `k`=0.25 to under 5% by `k`=1.6, and 0% at `k`=2.5.

`k`=0.25 and `k`=0.5 both had high prevelence of conflict escalation (42.1% and 37.0%) and collapse rates above 50%, making them poor definitions of true escalation. XXXX was chosen as the point at which collapse is effectively resolved while retaining strong genuine (non-collapsed) model performance.

In [12]:
prevalence_records = []

for k_test in [0.25, 0.5, 1, 1.25, 1.5, 1.6, 1.65, 1.75, 2, 2.5]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[], k=k_test, event_col="sub_event_type", conflict_only_embeddings=True,
    )
    onset_slice = model_data[
        (model_data["year_month"] >= pd.Period(ONSET_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ONSET_END_DATE, freq="M"))
    ]
    n_total = len(onset_slice)
    n_escalations = int(onset_slice["target_escalation"].sum())
    prevalence_records.append({
        "k": k_test,
        "onset_prevalence_pct": round(n_escalations / n_total * 100, 1),
        "n_escalations": n_escalations,
        "n_onset_rows": n_total,
    })

prevalence_df = pd.DataFrame(prevalence_records).set_index("k")
prevalence_df

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.25 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.5 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acle

,onset_prevalence_pct,n_escalations,n_onset_rows
k,,,
0.25,42.1,91,216
0.50,37.0,80,216
1.00,30.6,66,216
1.25,28.7,62,216
1.50,24.5,53,216
1.60,24.1,52,216
1.65,23.6,51,216
1.75,22.7,49,216
2.00,20.4,44,216


In [15]:
collapsed = sudan[sudan["onset_recall_class1"] == 1.0]
baseline_by_k = collapsed.groupby("k")["onset_precision_class1"].median()
collapse_rate = sudan.groupby("k")["onset_recall_class1"].apply(lambda x: (x == 1.0).mean())

k_summary = pd.DataFrame({
    "mean_onset_aupr": sudan.groupby("k")["onset_aupr"].mean(),
    "max_onset_aupr": sudan.groupby("k")["onset_aupr"].max(),
    "baseline_prevalence": baseline_by_k,
    "collapse_rate": collapse_rate,
})
k_summary.round(3)
k_summary = k_summary.join(prevalence_df[["onset_prevalence_pct"]])

In [27]:
ks = k_summary.index.astype(str)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Positive-class prevalence by k", "Threshold-collapse rate (%)"))

fig.add_trace(
    go.Bar(x=ks, y=k_summary["onset_prevalence_pct"], marker_color="#898781", name="Baseline prevalence"),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=ks, y=k_summary["collapse_rate"] * 100, marker_color="#c0392b", name="Collapse rate"),
    row=1, col=2,
)

fig.update_xaxes(title_text="k", row=1, col=1)
fig.update_xaxes(title_text="k", row=1, col=2)
fig.update_yaxes(title_text="% of onset rows that were escalations", row=1, col=1)
fig.update_yaxes(title_text="% of runs with recall=1.0", row=1, col=2)

fig.update_layout(
    showlegend=False, width=900, height=450,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(t=120),
    title={
        "text": "Lower escalation thresholds (k) had higher prevalence but were far more prone<br>to threshold collapse",
        "y": 0.9, "yanchor": "top",
    },
)
fig.show()

## 2. The threshold-collapse bug and fix

**Bug**

In the initial set of runs `optimal_threshold = thresholds[np.argmax(f1_scores)]` picked the lowest tied
threshold whenever F1 plateaued, producing "predict everything positive" models. This was disovered because many of the runs had recall=1 and precision=the actual proportion of conflict. 

**Fix**

Instead the model is set to pick the highest threshold when F1s were the same, the model becomes more conservative and less likely to predict everything positive.
```python
max_f1 = f1_scores.max()
tied_indices = np.flatnonzero(f1_scores == max_f1)
optimal_threshold = thresholds[tied_indices[-1]]
```

The current dataset (`sudan_results.csv`) was generated after this fix.